In [44]:
from pathlib import Path
SOURCE = "sportsapipro.com"

# API endpoint name used in the research document
ENDPOINT = "teams/players"

# Full API URL
URL = "https://api.sportsapipro.com/v2/football/teams/30/players"

# HTTP method
METHOD = "GET"
API_KEY_ENV_VAR = "API_FOOTBALL_KEY"

# API-Football uses this header
AUTH_HEADER_NAME = "x-api-key"

PARAMS = {
}

RAW_JSON_FILE = Path(
    "raw/sportsapipro/seasons/match/players.json"
)

OUTPUT_FILE = Path(
    "research/sportsapipro.md"
)

NOTES = """
Premier League 2025/26 season.
Initial investigation of the players endpoint.
"""

In [45]:
import os
import json
from datetime import datetime, timezone
from typing import Any
from dotenv import load_dotenv
import requests
load_dotenv()
def get_value_type(value: Any) -> str:

    if value is None:
        return "null"

    if isinstance(value, bool):
        return "boolean"

    if isinstance(value, int):
        return "integer"

    if isinstance(value, float):
        return "number"

    if isinstance(value, str):
        return "string"

    if isinstance(value, list):
        return "array"

    if isinstance(value, dict):
        return "object"

    return type(value).__name__


def collect_structure(
    value: Any,
    path: str = "$",
    rows: list | None = None,
) -> list:

    if rows is None:
        rows = []

    if isinstance(value, dict):

        for key, child in value.items():

            child_path = f"{path}.{key}"

            rows.append(
                {
                    "path": child_path,
                    "type": get_value_type(child),
                }
            )

            collect_structure(
                child,
                child_path,
                rows,
            )

    elif isinstance(value, list):

        if value:

            item = value[0]

            item_path = f"{path}[]"

            rows.append(
                {
                    "path": item_path,
                    "type": get_value_type(item),
                }
            )

            collect_structure(
                item,
                item_path,
                rows,
            )

    return rows


def format_params(params: dict) -> str:

    if not params:
        return "_No query parameters._"

    lines = [
        "| Parameter | Value |",
        "|---|---|",
    ]

    for key, value in params.items():

        if isinstance(value, (dict, list)):
            value = json.dumps(
                value,
                ensure_ascii=False,
            )

        value = str(value).replace("|", "\\|")

        lines.append(
            f"| `{key}` | `{value}` |"
        )

    return "\n".join(lines)


def format_structure(structure: list) -> str:

    if not structure:
        return "_No nested structure detected._"

    lines = [
        "| JSON Path | Type |",
        "|---|---|",
    ]

    seen = set()

    for row in structure:

        key = (
            row["path"],
            row["type"],
        )

        if key in seen:
            continue

        seen.add(key)

        lines.append(
            f"| `{row['path']}` | `{row['type']}` |"
        )

    return "\n".join(lines)


def get_response_summary(data: Any) -> str:

    lines = []

    if isinstance(data, dict):

        if "get" in data:
            lines.append(
                f"- **API endpoint reported:** `{data['get']}`"
            )

        if "results" in data:
            lines.append(
                f"- **Results:** `{data['results']}`"
            )

        if "paging" in data:

            lines.append("- **Paging:**")
            lines.append("```json")

            lines.append(
                json.dumps(
                    data["paging"],
                    indent=2,
                    ensure_ascii=False,
                )
            )

            lines.append("```")

        if "count" in data:
            lines.append(
                f"- **Count:** `{data['count']}`"
            )

        if "totalElements" in data:
            lines.append(
                f"- **Total elements:** `{data['totalElements']}`"
            )

        if "totalPages" in data:
            lines.append(
                f"- **Total pages:** `{data['totalPages']}`"
            )

        if "errors" in data:

            lines.append("- **Errors:**")
            lines.append("```json")

            lines.append(
                json.dumps(
                    data["errors"],
                    indent=2,
                    ensure_ascii=False,
                )
            )

            lines.append("```")

    if not lines:
        return "_No standard response metadata detected._"

    return "\n".join(lines)


def get_record_count(data: Any) -> str:

    if not isinstance(data, dict):
        return "Top-level response is not an object."

    counts = []

    for key, value in data.items():

        if isinstance(value, list):

            counts.append(
                f"- `{key}`: **{len(value)} records**"
            )

    if counts:
        return "\n".join(counts)

    return "_No top-level arrays detected._"


def get_representative_examples(
    data: Any,
    max_examples: int = 2,
) -> str:

    if not isinstance(data, dict):
        return "_No representative records available._"

    for key, value in data.items():

        if isinstance(value, list) and value:

            examples = value[:max_examples]

            return f"""### `{key}`

Showing {len(examples)} representative record(s).

```json
{json.dumps(
    examples,
    indent=2,
    ensure_ascii=False,
)}
```"""

    return "_No top-level record array detected._"

api_key = os.getenv(API_KEY_ENV_VAR)

if not api_key:

    raise RuntimeError(
        f"""
API key not found.

Set the environment variable:

{API_KEY_ENV_VAR}

Then restart your Jupyter kernel and run the notebook again.
"""
    )

headers = {
    AUTH_HEADER_NAME: api_key,
    "Accept": "application/json",
}


print(f"Calling: {URL}")
print(f"Parameters: {PARAMS}")

response = requests.request(
    method=METHOD,
    url=URL,
    params=PARAMS,
    headers=headers,
    timeout=30,
)

STATUS_CODE = response.status_code

print(f"HTTP Status: {STATUS_CODE}")

response.raise_for_status()


try:

    response_data = response.json()

except ValueError as exc:

    raise ValueError(
        "API response was not valid JSON."
    ) from exc


RAW_JSON_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with RAW_JSON_FILE.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        response_data,
        file,
        indent=2,
        ensure_ascii=False,
    )


print(f"Raw response saved to: {RAW_JSON_FILE}")

structure = collect_structure(
    response_data
)

captured_at = datetime.now(
    timezone.utc
).isoformat()

response_summary = get_response_summary(
    response_data
)

record_count = get_record_count(
    response_data
)

examples = get_representative_examples(
    response_data
)

structure_markdown = format_structure(
    structure
)

params_markdown = format_params(
    PARAMS
)


OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not OUTPUT_FILE.exists():

    header = f"""# {SOURCE} — API Investigation

## Project Context

- **Focus:** Premier League
- **Season:** 2025/26
- **Purpose:** Investigate API response structures and determine
  which data is useful for the football data pipeline.
- **Raw responses:** Stored separately as JSON files.

This document contains the investigation of multiple endpoints
from `{SOURCE}`.

"""

    OUTPUT_FILE.write_text(
        header,
        encoding="utf-8",
    )


existing_content = OUTPUT_FILE.read_text(
    encoding="utf-8"
)

endpoint_marker = f"# Endpoint: `{ENDPOINT}`"

if endpoint_marker in existing_content:

    raise ValueError(
        f"""
Endpoint already exists in:

{OUTPUT_FILE}

Endpoint:
{ENDPOINT}

The script will not overwrite existing research.
"""
    )


raw_file_display = str(
    RAW_JSON_FILE
).replace("\\", "/")


endpoint_section = f"""
---

# Endpoint: `{ENDPOINT}`

## 1. Request

| Property | Value |
|---|---|
| Source | `{SOURCE}` |
| Endpoint | `{ENDPOINT}` |
| Method | `{METHOD}` |
| URL | `{URL}` |
| HTTP Status | `{STATUS_CODE}` |
| Captured At | `{captured_at}` |
| Raw Response | `{raw_file_display}` |

### Query Parameters

{params_markdown}

### Notes

{NOTES.strip()}

---

## 2. Response Metadata

{response_summary}

---

## 3. Record Counts

{record_count}

---

## 4. Response Structure

The following structure is automatically derived from the
complete JSON response.

{structure_markdown}

---

## 5. Representative Response Examples

{examples}

---

## 6. Initial Investigation

### Data availability

- [ ] Does this endpoint contain the data we expected?
- [ ] Is the previous Premier League season fully represented?
- [ ] Are there missing or null fields?
- [ ] Is pagination present?
- [ ] Are there API-specific limits?

### Data modelling

- [ ] What are the stable identifiers?
- [ ] Which fields represent entities?
- [ ] Which fields represent relationships?
- [ ] Which nested objects should become separate entities/tables?
- [ ] Which fields overlap with the other API?
- [ ] Which fields are unique to this source?

### Pipeline relevance

- [ ] Is this endpoint required?
- [ ] Is this endpoint a source of truth for any canonical field?
- [ ] Does it need to be called once per season?
- [ ] Does it need to be called once per team?
- [ ] Does it need to be called once per match?
- [ ] How many requests would a complete season require?
- [ ] Can the response be cached and reused?

### Cross-source mapping

- [ ] What provider IDs need canonical mapping?
- [ ] Can entities be matched deterministically?
- [ ] Are there naming differences between providers?
- [ ] Are there provider-specific fields we need to preserve?

### Decisions / observations

_Add conclusions here after inspecting this endpoint._

"""


with OUTPUT_FILE.open(
    "a",
    encoding="utf-8",
) as file:

    file.write(
        endpoint_section
    )


print()
print("=" * 60)
print("SUCCESS")
print("=" * 60)
print(f"API:        {SOURCE}")
print(f"Endpoint:   {ENDPOINT}")
print(f"HTTP:       {STATUS_CODE}")
print(f"Raw JSON:   {RAW_JSON_FILE}")
print(f"Research:   {OUTPUT_FILE}")
print("=" * 60)

Calling: https://api.sportsapipro.com/v2/football/teams/30/players
Parameters: {}
HTTP Status: 200
Raw response saved to: raw\sportsapipro\seasons\match\players.json

SUCCESS
API:        sportsapipro.com
Endpoint:   teams/players
HTTP:       200
Raw JSON:   raw\sportsapipro\seasons\match\players.json
Research:   research\sportsapipro.md


In [8]:
import requests

headers = {'X-Auth-Token': 'a111f46d23064a3baef62b183596e375'}

# Step 1: Discover seasons
params = {
    "season": 2025,
    "matchday": 20
}
match = requests.get(
    'https://api.football-data.org/v4/competitions/PL/scorers/',
    headers=headers,
    params=params
).json()
match

{'count': 10,
 'filters': {'season': 2025, 'limit': 10},
 'competition': {'id': 2021,
  'name': 'Premier League',
  'code': 'PL',
  'type': 'LEAGUE',
  'emblem': 'https://crests.football-data.org/PL.png'},
 'season': {'id': 2403,
  'startDate': '2025-08-15',
  'endDate': '2026-05-24',
  'currentMatchday': 38,
  'winner': None},
 'scorers': [{'player': {'id': 38101,
    'name': 'Erling Haaland',
    'firstName': 'Erling',
    'lastName': 'Haaland',
    'dateOfBirth': '2000-07-21',
    'nationality': 'Norway',
    'section': 'Offence',
    'position': None,
    'shirtNumber': None,
    'lastUpdated': '2026-06-01T11:16:30Z'},
   'team': {'id': 65,
    'name': 'Manchester City FC',
    'shortName': 'Man City',
    'tla': 'MCI',
    'crest': 'https://crests.football-data.org/65.png',
    'address': 'SportCity Manchester M11 3FF',
    'website': 'https://www.mancity.com',
    'founded': 1880,
    'clubColors': 'Sky Blue / White',
    'venue': 'Etihad Stadium',
    'lastUpdated': '2022-02-10T

In [ ]:
import soccerdata as sd
fbref = sd.FBref(leagues="ENG-Premier League", seasons=2025)
# players_df = fbref.read_player_season_stats(stat_type="standard")
# players_df.head()
match_stats_df = fbref.read_player_match_stats(stat_type="summary")
match_stats_df.head()

In [2]:
from pathlib import Path
import re

INPUT_FILE = Path(
    "research/sportsapipro.md"
)

# Cleaned Markdown output.
OUTPUT_FILE = Path(
    "research/sportsapipro_clean.md"
)



if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input Markdown file not found:\n{INPUT_FILE.resolve()}"
    )

text = INPUT_FILE.read_text(
    encoding="utf-8"
)


text = re.sub(
    r"(?ms)^## 5\. Representative Response Examples\s*\n.*?(?=^## |\Z)",
    "",
    text,
)


# ============================================================
# REMOVE SECTION 6
#
# Removes:
#
# ## 6. Initial Investigation
#
# Everything until the next ## heading or end of file.
# ============================================================

text = re.sub(
    r"(?ms)^## 6\. Initial Investigation\s*\n.*?(?=^## |\Z)",
    "",
    text,
)


# ============================================================
# REMOVE ### NOTES SECTIONS
#
# Removes:
#
# ### Notes
#
# Everything until the next heading of the same or higher level.
#
# This handles Notes appearing under different endpoint sections.
# ============================================================

text = re.sub(
    r"(?ms)^### Notes\s*\n.*?(?=^### |^## |^# |\Z)",
    "",
    text,
)


# ============================================================
# CLEAN EXCESSIVE BLANK LINES
# ============================================================

text = re.sub(
    r"\n{3,}",
    "\n\n",
    text,
)


# Keep the document clean at the beginning/end.
text = text.strip() + "\n"


# ============================================================
# SAVE CLEAN FILE
# ============================================================

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_FILE.write_text(
    text,
    encoding="utf-8",
)


# ============================================================
# RESULT
# ============================================================

print("=" * 60)
print("MARKDOWN CLEANUP COMPLETE")
print("=" * 60)
print(f"Input :  {INPUT_FILE.resolve()}")
print(f"Output:  {OUTPUT_FILE.resolve()}")
print()
print("Removed:")
print("  - ## 5. Representative Response Examples")
print("  - ## 6. Initial Investigation")
print("  - ### Notes sections")
print("=" * 60)

MARKDOWN CLEANUP COMPLETE
Input :  C:\Users\rahul\Documents\Projects\Football-data\research\sportsapipro.md
Output:  C:\Users\rahul\Documents\Projects\Football-data\research\sportsapipro_clean.md

Removed:
  - ## 5. Representative Response Examples
  - ## 6. Initial Investigation
  - ### Notes sections


Load Football-data.org data

Leagues

In [ ]:
import requests
import json
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()

# ---- Configure these ----
URL = "https://api.football-data.org/v4/competitions/PL"
METHOD = "GET"
HEADERS = {
    "X-Auth-Token": os.getenv("FOOTBALL_DATA_ORG_API_KEY"),
}
PARAMS = {
}
OUTPUT_PATH = Path("raw/2025/football_data_org/leagues.json")
response = requests.request(
    method=METHOD,
    url=URL,
    headers=HEADERS,
    params=PARAMS,
)
response.raise_for_status()
data = response.json()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Saved response to {OUTPUT_PATH.resolve()}")

Standings

In [ ]:
import requests
import json
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()

# ---- Configure these ----
URL = "https://api.football-data.org/v4/competitions/PL/stadings"
METHOD = "GET"
HEADERS = {
    "X-Auth-Token": os.getenv("FOOTBALL_DATA_ORG_API_KEY"),
}
PARAMS = {
    "season": 2025
}
OUTPUT_PATH = Path("raw/2025/football_data_org/stadings.json")
response = requests.request(
    method=METHOD,
    url=URL,
    headers=HEADERS,
    params=PARAMS,
)
response.raise_for_status()
data = response.json()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Saved response to {OUTPUT_PATH.resolve()}")

Top scores of entire Season

In [3]:
import requests
import json
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()

# ---- Configure these ----
URL = "https://api.football-data.org/v4/competitions/PL/scorers/"
METHOD = "GET"
HEADERS = {
    "X-Auth-Token": os.getenv("FOOTBALL_DATA_ORG_API_KEY"),
}
PARAMS = {
    "season": 2025
}
OUTPUT_PATH = Path("raw/2025/football_data_org/scorers.json")
response = requests.request(
    method=METHOD,
    url=URL,
    headers=HEADERS,
    params=PARAMS,
)
response.raise_for_status()
data = response.json()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Saved response to {OUTPUT_PATH.resolve()}")

Saved response to C:\Users\rahul\Documents\Projects\Football-data\raw\2025\football_data_org\scorers.json


Matches

In [9]:
import requests
import json
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()

# ---- Configure these ----
URL = "https://api.football-data.org/v4/competitions/PL/matches"
METHOD = "GET"
HEADERS = {
    "X-Auth-Token": os.getenv("FOOTBALL_DATA_ORG_API_KEY"),
}
PARAMS = {
    "season": 2025
}
OUTPUT_PATH = Path("raw/2025/football_data_org/matches.json")
response = requests.request(
    method=METHOD,
    url=URL,
    headers=HEADERS,
    params=PARAMS,
)
response.raise_for_status()
data = response.json()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Saved response to {OUTPUT_PATH.resolve()}")

Saved response to C:\Users\rahul\Documents\Projects\Football-data\raw\2025\football_data_org\matches.json


Teams

In [11]:
import requests
import json
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()

# ---- Configure these ----
URL = "https://api.football-data.org/v4/competitions/PL/teams"
METHOD = "GET"
HEADERS = {
    "X-Auth-Token": os.getenv("FOOTBALL_DATA_ORG_API_KEY"),
}
PARAMS = {
    "season": 2025
}
OUTPUT_PATH = Path("raw/2025/football_data_org/teams.json")
response = requests.request(
    method=METHOD,
    url=URL,
    headers=HEADERS,
    params=PARAMS,
)
response.raise_for_status()
data = response.json()
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Saved response to {OUTPUT_PATH.resolve()}")

Saved response to C:\Users\rahul\Documents\Projects\Football-data\raw\2025\football_data_org\teams.json
